In [1]:
import pandas as pd

# 1)

In [2]:
loans_raw = pd.read_csv('../data/library_loans_clean.csv', parse_dates=['loan_date','return_date'])
loans = loans_raw.copy()
print(loans.shape)
print(loans.dtypes)

(600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object


In [3]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   loan_id      600 non-null    str           
 1   member_id    600 non-null    str           
 2   title        600 non-null    str           
 3   genre        600 non-null    str           
 4   loan_date    597 non-null    datetime64[us]
 5   return_date  564 non-null    datetime64[us]
 6   late_fee     600 non-null    float64       
 7   returned     600 non-null    bool          
dtypes: bool(1), datetime64[us](2), float64(1), str(4)
memory usage: 33.5 KB


In [4]:
print(loans.loan_date.isna().mean()*100,'% ', loans.loan_date.isna().sum())
print(loans.return_date.isna().mean()*100,'% ', loans.return_date.isna().sum())

0.5 %  3
6.0 %  36


# 2)

In [5]:
loans[loans.loan_date.isna()]

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
462,L-30463,M0007,Blue Moon Protocol,Sci-fi,NaT,2026-05-24,0.0,True
467,L-30468,M0018,Blue Moon Protocol,Sci-fi,NaT,2026-06-08,0.0,True
571,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True


In [6]:
loans[loans.return_date.isna()].groupby('returned')['loan_id'].count()

returned
False    30
True      6
Name: loan_id, dtype: int64

if it hasn't been returned yet then the return_date is missing for a reason
if it has been returned and the return_date is still missing, then it's missing by accident

In [7]:
loans['has_return_date'] = loans.return_date.notna()

In [8]:
print(loans.has_return_date.sum())

564


# 3)

In [9]:
members_raw = pd.read_csv('../data/library_members.csv')
members = members_raw.copy()
print(members.shape)

(148, 3)


In [10]:
members.info()

<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB


In [11]:
members.dtypes

member_id         str
branch            str
birth_year    float64
dtype: object

In [12]:
members.describe()

,birth_year
count,130.000000
mean,1983.384615
std,25.158223
min,1900.000000
25%,1978.250000
50%,1988.000000
75%,1996.750000
max,2030.000000


In [13]:
print(members.loc[members['birth_year'] == 1900]['member_id'].count())
print(members.loc[members['birth_year'] == 2030]['member_id'].count())
print(members.duplicated().sum())
print(members['member_id'].duplicated().sum())

8
3
4
4


In [14]:
members[members.duplicated(keep=False)]

,member_id,branch,birth_year
13,M0015,Central,1995.0
14,M0015,Central,1995.0
31,M0033,South,1975.0
32,M0033,South,1975.0
63,M0065,Central,1988.0
64,M0065,Central,1988.0
68,M0071,Central,1970.0
69,M0071,Central,1970.0


Problems:
* 18 missing birth_years
* 8 birth_years are 1900
* 3 birth_years are 2030
* 4 members are duplicates

# 4)

In [15]:
members.birth_year = members.birth_year.where(members.birth_year.between(1901,2026))

In [16]:
members.describe()

,birth_year
count,119.000000
mean,1987.815126
std,12.062666
min,1957.000000
25%,1979.000000
50%,1988.000000
75%,1996.500000
max,2010.000000


In [17]:
print(members.birth_year.isna().sum())

29


# 5 a)

In [18]:
print(members.dropna().shape)
print(members.dropna(subset=['branch']).shape)
print(members.dropna(subset=['birth_year']).shape)

(107, 3)
(136, 3)
(119, 3)


# b)

In [19]:
members.branch = members.branch.fillna('Unknown')

In [20]:
print(members.branch.count())
members.groupby('branch').count().sort_values('member_id', ascending=False)

148


,member_id,birth_year
branch,,
Central,40,34
South,34,27
North,32,21
East,17,12
Harbour,13,13
Unknown,12,12


# c)

In [21]:
median_birth_year = members.birth_year.median()
median_fill = members.birth_year.fillna(median_birth_year)

In [22]:
print(members.birth_year.mean().round(1), members.birth_year.std().round(1))
print(median_fill.mean().round(1), median_fill.std().round(1))

1987.8 12.1
1987.9 10.8


Adding a bunch of values that are all median values would decrease the standard deviation.
It doesn't make sense to add birth_years that we are just guessing blindly at. Better to leave the birth_year as NaN

# 6)

In [23]:
print(loans.shape)
print(members.shape)

(600, 9)
(148, 3)


In [24]:
merged = loans.merge(members, how='left')
merged.shape

(616, 11)

In [25]:
merged[merged.duplicated(keep=False)].head(6)

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,has_return_date,branch,birth_year
29,L-30030,M0071,Blue Moon Protocol,Sci-fi,2026-01-14,2026-01-19,0.0,True,True,Central,1970.0
30,L-30030,M0071,Blue Moon Protocol,Sci-fi,2026-01-14,2026-01-19,0.0,True,True,Central,1970.0
60,L-30060,M0071,The Locksmith,Crime,2026-01-22,2026-02-02,0.0,True,True,Central,1970.0
61,L-30060,M0071,The Locksmith,Crime,2026-01-22,2026-02-02,0.0,True,True,Central,1970.0
144,L-30143,M0065,The Glass Mage,Fantasy,2026-02-15,2026-03-24,40.0,True,True,Central,1988.0
145,L-30143,M0065,The Glass Mage,Fantasy,2026-02-15,2026-03-24,40.0,True,True,Central,1988.0


In [26]:
members[members.duplicated(keep=False)].head(6)

,member_id,branch,birth_year
13,M0015,Central,1995.0
14,M0015,Central,1995.0
31,M0033,South,1975.0
32,M0033,South,1975.0
63,M0065,Central,1988.0
64,M0065,Central,1988.0


merged table has duplicates since members still has duplicates

In [27]:
print(loans.late_fee.sum())
print(merged.late_fee.sum())

7487.5
7725.0


In [28]:
loans.merge(members, how='left', validate='many_to_one')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

# 7)

In [29]:
members = members.drop_duplicates().reset_index(drop=True)
members.shape

(144, 3)

In [30]:
merged = loans.merge(members, how='left', validate='many_to_one', indicator=True)

In [31]:
merged.shape

(600, 12)

In [32]:
print(merged.late_fee.sum())

7487.5


In [33]:
merged._merge.value_counts()

_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64

In [34]:
merged.head(3)

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,has_return_date,branch,birth_year,_merge
0,L-30001,M0150,A Second Spring,Romance,2026-01-06,2026-01-30,7.5,True,True,Central,1994.0,both
1,L-30002,M0037,The Silent Harbour,Crime,2026-01-05,2026-01-25,0.0,True,True,South,1987.0,both
2,L-30003,M0143,The Glass Mage,Fantasy,2026-01-05,2026-01-26,0.0,True,True,Harbour,1976.0,both


In [35]:
print(len(merged.where(merged._merge == 'left_only').groupby('member_id')))

6


In [36]:
print(merged.where(merged._merge == 'left_only')['late_fee'].sum())

347.5


In [37]:
inner = loans.merge(members, how='inner', validate='many_to_one')
inner.shape

(569, 11)

In [38]:
print(inner.late_fee.sum())

7140.0


# 8)

In [39]:
merged.branch = merged.branch.fillna('no member record')

In [40]:
merged.groupby('branch')['loan_id'].count().sort_values(ascending=False)

branch
Central             144
South               130
North               126
East                 72
Harbour              51
Unknown              46
no member record     31
Name: loan_id, dtype: int64

Unknown referes to members that exist in the members table, but with unknown branch.
'no member record' means the loan member_id doesn't have a matching member_id in the members table.

In [42]:
merged.to_csv('../data/library_loans_clean_merged.csv', index=False)
members.to_csv('../data/members_clean.csv', index=False)